# 株・為替 取引シミュレーター

yfinance（Yahoo Finance）の価格を使って、**キオクシア・レーザーテック・ドル円・ユーロ円**のロング（買い）とショート（売り）を仮想的に取引し、損益を記録します。

## 使い方

1. メニューの **ランタイム → すべてのセルを実行** を選びます（または上から順に ▶ を押します）。
2. Google ドライブへのアクセスを求められたら許可してください。取引データはドライブの `trade_sim/portfolio.json` に保存され、次に開いたときも続きから使えます。
3. ②のセルの下に表示される画面で、銘柄・売買・数量を選んで **発注する** を押します。決済は「保有中の建玉」で行います。

- 約定日（YYYY-MM-DD）を入れると、その日の終値で約定します（過去の価格でのシミュレーション）。空欄なら最新の価格です。
- 価格は Yahoo Finance のデータです。東証の銘柄はおよそ 20 分遅れです。
- 手数料、スプレッド、スワップポイント、信用取引の金利・貸株料は計算に含めていません。

In [ ]:
#@title ① セットアップ（最初に実行してください） { display-mode: "form" }
#@markdown **save_to_drive**: チェックを入れておくと取引データを Google ドライブに保存し、次に開いたときも続きから使えます。
save_to_drive = True  #@param {type:"boolean"}

import importlib.metadata
import re
import subprocess
import sys
from pathlib import Path


def _version(package):
    try:
        return tuple(int(x) for x in re.findall(r"\d+", importlib.metadata.version(package))[:3])
    except importlib.metadata.PackageNotFoundError:
        return ()


# yfinance が入っていないか古ければ、最新版を入れる
if _version("yfinance") < (1, 7, 0):
    print("yfinance をインストールしています…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance"], check=True)

# シミュレーター本体 (trade_sim パッケージ) を書き出す
_SOURCES = {
    'trade_sim/__init__.py': r'''"""yfinance の価格を使った株・為替の取引シミュレーター。"""

__version__ = "0.1.0"
''',
    'trade_sim/__main__.py': r'''from .cli import main

raise SystemExit(main())
''',
    'trade_sim/cli.py': r'''"""コマンドラインのエントリーポイント。 `python -m trade_sim --help` で使い方を表示する。"""

from __future__ import annotations

import argparse
import os
import sys
from datetime import date, datetime

from . import __version__
from .instruments import INSTRUMENTS
from .portfolio import LONG, SHORT
from .prices import PriceSource
from .simulator import USER_ERRORS, Result, Simulator, now_jst, parse_date
from .table import render_table

DEFAULT_FILE = "portfolio.json"
FILE_ENV = "TRADE_SIM_FILE"


def print_result(result: Result) -> None:
    if result.table is not None:
        table = result.table
        print(render_table(table.headers, table.rows, table.aligns, table.footer))
        if result.lines or result.notes:
            print()
    for line in result.lines:
        print(line)
    for note in result.notes:
        print(f"※ {note}")


# ---- コマンド ------------------------------------------------------------


def cmd_quote(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.quote(args.symbols)


def cmd_open(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.open(args.symbol, args.side, args.qty, price=args.price, day=args.date)


def cmd_close(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.close(args.ids, qty=args.qty, price=args.price, day=args.date)


def cmd_positions(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.positions()


def cmd_history(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.history(args.symbol)


def cmd_summary(args: argparse.Namespace, sim: Simulator) -> Result:
    return sim.summary()


def cmd_reset(args: argparse.Namespace, sim: Simulator) -> Result:
    if sim.path.exists() and not args.yes:
        try:
            answer = input(f"{sim.path} の建玉と取引履歴をすべて削除します。よろしいですか? [y/N]: ")
        except EOFError:
            answer = ""
        if answer.strip().lower() not in ("y", "yes"):
            return Result(lines=["中止しました。"])
    return sim.reset()


# ---- 引数の解析 ----------------------------------------------------------


def _positive_int(text: str) -> int:
    try:
        value = int(text.replace(",", ""))
    except ValueError:
        raise argparse.ArgumentTypeError(f"整数で指定してください: {text}") from None
    if value <= 0:
        raise argparse.ArgumentTypeError(f"1 以上を指定してください: {text}")
    return value


def _positive_float(text: str) -> float:
    try:
        value = float(text.replace(",", ""))
    except ValueError:
        raise argparse.ArgumentTypeError(f"数値で指定してください: {text}") from None
    if not value > 0:
        raise argparse.ArgumentTypeError(f"0 より大きい値を指定してください: {text}")
    return value


def _date(text: str) -> date:
    try:
        day = parse_date(text)
    except USER_ERRORS as e:
        raise argparse.ArgumentTypeError(str(e)) from None
    if day is None:
        raise argparse.ArgumentTypeError("日付を YYYY-MM-DD 形式で指定してください")
    return day


def _add_price_options(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--date",
        type=_date,
        metavar="YYYY-MM-DD",
        help="その日の終値で約定させる (休場日なら直前の営業日)。省略すると最新の価格",
    )
    parser.add_argument("--price", type=_positive_float, metavar="PRICE", help="約定価格を直接指定する")


def _epilog() -> str:
    lines = ["対応銘柄 (銘柄名・コード・ティッカーのどれでも指定できます):"]
    for inst in INSTRUMENTS:
        names = " / ".join(dict.fromkeys([inst.name, inst.key.lower(), *inst.aliases]))
        lines.append(f"  {names}  [{inst.ticker}]")
    lines += [
        "",
        "例:",
        "  python -m trade_sim quote                       現在値を表示",
        "  python -m trade_sim long kioxia 100             キオクシアを 100 株ロング",
        "  python -m trade_sim short usdjpy 10000          ドル円を 1 万通貨ショート",
        "  python -m trade_sim long 6920 100 --date 2025-04-01   過去の終値で約定",
        "  python -m trade_sim positions                   保有中の建玉と評価損益",
        "  python -m trade_sim close 1                     建玉 #1 を決済",
        "  python -m trade_sim summary                     通算損益を表示",
    ]
    return "\n".join(lines)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="python -m trade_sim",
        description="yfinance の価格で、株・為替のロング/ショート取引をシミュレーションします。",
        epilog=_epilog(),
        formatter_class=argparse.RawDescriptionHelpFormatter,
    )
    parser.add_argument(
        "--file",
        default=os.environ.get(FILE_ENV, DEFAULT_FILE),
        metavar="PATH",
        help=f"取引データの保存先 (既定: {DEFAULT_FILE}。環境変数 {FILE_ENV} でも指定可)",
    )
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
    sub = parser.add_subparsers(dest="command", metavar="COMMAND")

    p = sub.add_parser("quote", help="現在値を表示する")
    p.add_argument("symbols", nargs="*", metavar="SYMBOL", help="銘柄 (省略すると全銘柄)")
    p.set_defaults(func=cmd_quote)

    for side, text in ((LONG, "新規にロング (買い) で建てる"), (SHORT, "新規にショート (売り) で建てる")):
        p = sub.add_parser(side, help=text, description=text)
        p.add_argument("symbol", metavar="SYMBOL", help="銘柄 (例: kioxia、6920、ドル円、eurjpy)")
        p.add_argument("qty", type=_positive_int, metavar="QTY", help="数量。株数または通貨数 (例: 100、10000)")
        _add_price_options(p)
        p.set_defaults(func=cmd_open, side=side)

    p = sub.add_parser("close", help="建玉を決済する", description="建玉を決済する")
    p.add_argument("ids", nargs="+", metavar="ID", help="決済する建玉の ID (複数可)。all ですべての建玉")
    p.add_argument("--qty", type=_positive_int, metavar="QTY", help="一部だけ決済する数量")
    _add_price_options(p)
    p.set_defaults(func=cmd_close)

    p = sub.add_parser("positions", help="保有中の建玉と評価損益を表示する")
    p.set_defaults(func=cmd_positions)

    p = sub.add_parser("history", help="決済済みの取引履歴を表示する")
    p.add_argument("symbol", nargs="?", metavar="SYMBOL", help="銘柄 (指定するとその銘柄だけ表示)")
    p.set_defaults(func=cmd_history)

    p = sub.add_parser("summary", aliases=["pnl"], help="銘柄別と通算の損益を表示する")
    p.set_defaults(func=cmd_summary)

    p = sub.add_parser("reset", help="建玉と取引履歴をすべて削除する")
    p.add_argument("-y", "--yes", action="store_true", help="確認せずに削除する")
    p.set_defaults(func=cmd_reset)

    return parser


def main(
    argv: list[str] | None = None,
    source: PriceSource | None = None,
    now: datetime | None = None,
) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.command is None:
        parser.print_help()
        return 0
    clock = (lambda: now) if now is not None else now_jst
    sim = Simulator(args.file, source, clock)
    try:
        result = args.func(args, sim)
    except USER_ERRORS as e:
        print(f"エラー: {e}", file=sys.stderr)
        return 1
    print_result(result)
    return 0 if result.ok else 1
''',
    'trade_sim/instruments.py': r'''"""シミュレーション対象の銘柄の定義。

銘柄を増やすときは INSTRUMENTS に Instrument を追加する。
損益はすべて円で計算するので、円建ての銘柄 (東証の株、XXX/JPY の為替) を追加すること。
"""

from __future__ import annotations

import unicodedata
from dataclasses import dataclass


@dataclass(frozen=True)
class Instrument:
    key: str  # 内部 ID (保存データにはこの値が入る)
    name: str  # 表示名
    ticker: str  # yfinance (Yahoo Finance) のティッカー
    kind: str  # "stock" (株式) / "fx" (為替)
    unit: str  # 数量の単位
    price_decimals: int  # 価格の小数点以下の桁数
    aliases: tuple[str, ...] = ()  # コマンドで指定するときの別名

    def format_price(self, price: float) -> str:
        return f"{price:,.{self.price_decimals}f}"

    def format_qty(self, qty: int) -> str:
        return f"{qty:,}{self.unit}"


INSTRUMENTS: tuple[Instrument, ...] = (
    Instrument("KIOXIA", "キオクシア", "285A.T", "stock", "株", 1, ("285A",)),
    Instrument("LASERTEC", "レーザーテック", "6920.T", "stock", "株", 1, ("6920",)),
    Instrument("USDJPY", "ドル円", "USDJPY=X", "fx", "通貨", 3, ("USD/JPY", "ドル/円")),
    Instrument("EURJPY", "ユーロ円", "EURJPY=X", "fx", "通貨", 3, ("EUR/JPY", "ユーロ/円")),
)


class UnknownInstrumentError(ValueError):
    pass


def _normalize(text: str) -> str:
    # 全角英数字や半角カナでも指定できるように NFKC で正規化する
    return unicodedata.normalize("NFKC", text).strip().lower()


_BY_KEY = {inst.key: inst for inst in INSTRUMENTS}
_LOOKUP = {
    _normalize(name): inst
    for inst in INSTRUMENTS
    for name in (inst.key, inst.name, inst.ticker, *inst.aliases)
}


def get_instrument(key: str) -> Instrument:
    """保存データ内の ID から銘柄を引く。"""
    try:
        return _BY_KEY[key]
    except KeyError:
        raise UnknownInstrumentError(f"未対応の銘柄 ID です: {key}") from None


def find_instrument(text: str) -> Instrument:
    """コマンドで入力された銘柄名・コード・別名から銘柄を探す。"""
    inst = _LOOKUP.get(_normalize(text))
    if inst is None:
        choices = "、".join(f"{i.name}({i.key.lower()})" for i in INSTRUMENTS)
        raise UnknownInstrumentError(f"銘柄「{text}」は対応していません。対応銘柄: {choices}")
    return inst
''',
    'trade_sim/notebook.py': r'''"""Google Colab / Jupyter Notebook 用の取引画面 (ipywidgets)。

ノートブックのセルで次のように実行すると、注文・決済・損益確認ができる画面が出る。

    from trade_sim.notebook import show
    app = show("portfolio.json")
"""

from __future__ import annotations

import html
from collections.abc import Callable

import ipywidgets as widgets
from IPython.display import display

from .instruments import INSTRUMENTS, get_instrument
from .portfolio import LONG, SHORT, SIDE_LABELS
from .prices import PriceSource
from .simulator import USER_ERRORS, Result, Simulator, Table, parse_date

# 銘柄を選んだときの数量の初期値
DEFAULT_QTY = {"stock": 100, "fx": 10000}

# Colab のライト/ダーク両方で読めるよう、文字色は指定せず半透明の線だけ使う
_LINE = "rgba(127, 127, 127, 0.35)"
_ERROR = "#d9534f"


# ---- HTML の組み立て -----------------------------------------------------


def table_html(table: Table) -> str:
    def cells(row: list[str], tag: str, extra: str = "") -> str:
        out = []
        for text, align in zip(row, table.aligns):
            style = (
                f"padding: 4px 10px; border-bottom: 1px solid {_LINE}; white-space: nowrap;"
                f" text-align: {'right' if align == '>' else 'left'};{extra}"
            )
            out.append(f'<{tag} style="{style}">{html.escape(text)}</{tag}>')
        return "".join(out)

    parts = [
        '<div style="overflow-x: auto; max-width: 100%;">',
        # color / font-size: inherit は、互換モードの表示でも表がまわりの文字色・大きさを引き継ぐようにするため
        '<table style="border-collapse: collapse; color: inherit; font-size: inherit;'
        ' font-variant-numeric: tabular-nums; margin: 2px 0 6px;">',
        f"<thead><tr>{cells(table.headers, 'th', ' font-weight: 600;')}</tr></thead>",
        "<tbody>",
        *(f"<tr>{cells(row, 'td')}</tr>" for row in table.rows),
        "</tbody>",
    ]
    if table.footer:
        footer_style = f" font-weight: 600; border-top: 2px solid {_LINE};"
        parts += ["<tfoot>", *(f"<tr>{cells(row, 'td', footer_style)}</tr>" for row in table.footer), "</tfoot>"]
    parts += ["</table>", "</div>"]
    return "".join(parts)


def result_html(result: Result, emphasize: bool = False) -> str:
    """操作結果を HTML にする。emphasize=True なら最初の行 (通算損益など) を大きく表示する。"""
    parts = []
    if result.table is not None:
        parts.append(table_html(result.table))
    for i, line in enumerate(result.lines):
        style = "font-size: 1.2em; font-weight: 700; margin: 4px 0;" if emphasize and i == 0 else "margin: 2px 0;"
        parts.append(f'<div style="{style}">{html.escape(line)}</div>')
    for note in result.notes:
        parts.append(f'<div style="margin: 2px 0; opacity: 0.75; font-size: 0.92em;">※ {html.escape(note)}</div>')
    return "".join(parts)


def error_html(message: str) -> str:
    return f'<div style="color: {_ERROR}; font-weight: 600; margin: 2px 0;">エラー: {html.escape(message)}</div>'


def _info_html(message: str) -> str:
    return f'<div style="margin: 2px 0; opacity: 0.75;">{html.escape(message)}</div>'


def _heading(text: str) -> widgets.HTML:
    style = f"font-size: 1.1em; font-weight: 700; margin: 16px 0 2px; padding-bottom: 2px; border-bottom: 1px solid {_LINE};"
    return widgets.HTML(f'<div style="{style}">{html.escape(text)}</div>')


def _row(*children: widgets.Widget) -> widgets.HBox:
    return widgets.HBox(list(children), layout=widgets.Layout(flex_flow="row wrap", align_items="center"))


# ---- 画面 ----------------------------------------------------------------


class TradeApp:
    def __init__(self, simulator: Simulator) -> None:
        self.sim = simulator
        label = {"description_width": "4.5em"}
        first = INSTRUMENTS[0]

        # 現在値
        self.status = widgets.HTML()
        self.quote_view = widgets.HTML()
        self.refresh_button = widgets.Button(description="価格を更新", icon="refresh")

        # 新規注文
        self.symbol = widgets.Dropdown(
            options=[(inst.name, inst.key) for inst in INSTRUMENTS], value=first.key, description="銘柄", style=label
        )
        self.side = widgets.ToggleButtons(
            options=[("ロング（買い）", LONG), ("ショート（売り）", SHORT)], value=LONG, description="売買", style=label
        )
        self.qty = widgets.IntText(
            value=DEFAULT_QTY[first.kind], description="数量", style=label, layout=widgets.Layout(width="15em")
        )
        self.unit = widgets.HTML(first.unit)
        self.order_date = widgets.Text(
            placeholder="空欄なら最新の価格（例: 2025-04-01）",
            description="約定日",
            style=label,
            layout=widgets.Layout(width="26em"),
        )
        self.order_button = widgets.Button(description="発注する", button_style="primary")
        self.order_message = widgets.HTML()

        # 決済
        self.positions_view = widgets.HTML()
        self.close_target = widgets.Dropdown(description="建玉", style=label, layout=widgets.Layout(width="30em"))
        self.close_qty = widgets.IntText(value=0, description="数量", style=label, layout=widgets.Layout(width="15em"))
        self.close_date = widgets.Text(
            placeholder="空欄なら最新の価格", description="決済日", style=label, layout=widgets.Layout(width="26em")
        )
        self.close_button = widgets.Button(description="決済する", button_style="warning")
        self.close_all_button = widgets.Button(description="すべて決済")
        self.close_message = widgets.HTML()

        # 損益・履歴
        self.summary_view = widgets.HTML()
        self.history_view = widgets.HTML()

        self._busy = False
        self._has_positions = False
        self._confirming_close_all = False

        self.symbol.observe(self._on_symbol_change, names="value")
        self.refresh_button.on_click(self._on_refresh)
        self.order_button.on_click(self._on_order)
        self.close_button.on_click(self._on_close)
        self.close_all_button.on_click(self._on_close_all)

        self.widget = widgets.VBox(
            [
                widgets.HTML('<div style="font-size: 1.4em; font-weight: 700;">株・為替 取引シミュレーター</div>'),
                self.status,
                _heading("損益"),
                self.summary_view,
                _heading("現在値"),
                self.quote_view,
                self.refresh_button,
                _heading("新規注文"),
                self.symbol,
                self.side,
                _row(self.qty, self.unit),
                self.order_date,
                self.order_button,
                self.order_message,
                _heading("保有中の建玉"),
                self.positions_view,
                self.close_target,
                _row(self.close_qty, widgets.HTML("0 ならすべて決済")),
                self.close_date,
                _row(self.close_button, self.close_all_button),
                self.close_message,
                _heading("取引履歴"),
                self.history_view,
            ]
        )
        self.refresh()

    def refresh(self) -> None:
        """表示をすべて最新の状態にする。"""
        views: list[tuple[widgets.HTML, Callable[[], str]]] = [
            (self.summary_view, lambda: result_html(self.sim.summary(), emphasize=True)),
            (self.quote_view, lambda: result_html(self.sim.quote())),
            (self.positions_view, lambda: result_html(self.sim.positions())),
            (self.history_view, lambda: result_html(self.sim.history())),
        ]
        for view, render in views:
            try:
                view.value = render()
            except USER_ERRORS as e:
                view.value = error_html(str(e))
        self._update_close_targets()
        self.status.value = _info_html(
            f"最終更新: {self.sim.clock():%Y-%m-%d %H:%M}（日本時間）　データ: {self.sim.path}"
        )

    # ---- イベント --------------------------------------------------------

    def _on_symbol_change(self, change: dict) -> None:
        inst = get_instrument(change["new"])
        self.qty.value = DEFAULT_QTY[inst.kind]
        self.unit.value = inst.unit

    def _on_refresh(self, _button: widgets.Button) -> None:
        self._cancel_close_all()
        self._set_busy(True)
        try:
            self.sim.refresh_prices()
            self.refresh()
        finally:
            self._set_busy(False)

    def _on_order(self, _button: widgets.Button) -> None:
        self._cancel_close_all()
        symbol, side, qty, day_text = self.symbol.value, self.side.value, self.qty.value, self.order_date.value
        self._run(self.order_message, lambda: self.sim.open(symbol, side, qty, day=parse_date(day_text)))

    def _on_close(self, _button: widgets.Button) -> None:
        self._cancel_close_all()
        target = self.close_target.value
        if target is None:
            self.close_message.value = error_html("決済する建玉を選んでください。")
            return
        qty = self.close_qty.value or None  # 0 ならすべて決済
        day_text = self.close_date.value
        self._run(self.close_message, lambda: self.sim.close([target], qty=qty, day=parse_date(day_text)))

    def _on_close_all(self, _button: widgets.Button) -> None:
        if not self._confirming_close_all:
            self._confirming_close_all = True
            self.close_all_button.description = "もう一度押すと全決済"
            self.close_all_button.button_style = "danger"
            self.close_message.value = _info_html(
                "保有中の建玉をすべて決済します。よければ「もう一度押すと全決済」を押してください。"
            )
            return
        self._cancel_close_all()
        day_text = self.close_date.value
        self._run(self.close_message, lambda: self.sim.close(["all"], day=parse_date(day_text)))

    # ---- 内部 ------------------------------------------------------------

    def _run(self, message: widgets.HTML, action: Callable[[], Result]) -> None:
        """ボタンの操作を実行して、結果かエラーを message に表示する。"""
        self._set_busy(True)
        message.value = _info_html("処理中…")
        try:
            result = action()
        except USER_ERRORS as e:
            message.value = error_html(str(e))
        except Exception as e:  # ボタンから呼ばれるので、想定外のエラーも画面に出す
            message.value = error_html(f"予期しないエラーが発生しました ({type(e).__name__}: {e})")
        else:
            message.value = result_html(result)
            self.refresh()
        finally:
            self._set_busy(False)

    def _cancel_close_all(self) -> None:
        if self._confirming_close_all:
            self._confirming_close_all = False
            self.close_all_button.description = "すべて決済"
            self.close_all_button.button_style = ""

    def _update_close_targets(self) -> None:
        try:
            positions = self.sim.load().positions
        except USER_ERRORS:
            positions = []
        options = []
        for p in positions:
            inst = get_instrument(p.symbol)
            label = (
                f"#{p.id} {inst.name} {SIDE_LABELS[p.side]} {inst.format_qty(p.qty)}"
                f" @ {inst.format_price(p.entry_price)}"
            )
            options.append((label, p.id))
        current = self.close_target.value
        self.close_target.options = options
        values = [value for _, value in options]
        if values:
            # 選んでいた建玉が残っていればそのまま、なければ先頭を選ぶ (空の状態から増えたときは未選択になるため)
            self.close_target.value = current if current in values else values[0]
        self._has_positions = bool(options)
        self._update_buttons()

    def _set_busy(self, busy: bool) -> None:
        self._busy = busy
        self._update_buttons()

    def _update_buttons(self) -> None:
        self.refresh_button.disabled = self._busy
        self.order_button.disabled = self._busy
        can_close = self._has_positions and not self._busy
        self.close_target.disabled = not self._has_positions
        self.close_button.disabled = not can_close
        self.close_all_button.disabled = not can_close


def _enable_colab_widgets() -> None:
    """Colab で ipywidgets 8 以降を使う場合は、Colab のカスタムウィジェット表示を有効にする。"""
    try:
        from google.colab import output  # type: ignore[import-not-found]
    except ImportError:
        return
    if int(widgets.__version__.split(".")[0]) >= 8:
        output.enable_custom_widget_manager()


def show(data_file: str = "portfolio.json", source: PriceSource | None = None) -> TradeApp:
    """取引画面を表示する。data_file は取引データの保存先。"""
    _enable_colab_widgets()
    app = TradeApp(Simulator(data_file, source))
    display(app.widget)
    return app
''',
    'trade_sim/portfolio.py': r'''"""建玉と決済履歴の管理、損益の計算。

損益はすべて円で計算する (対象銘柄はすべて円建てのため)。
  ロング:   (決済値 - 建値) × 数量
  ショート: (建値 - 決済値) × 数量
"""

from __future__ import annotations

import json
import os
from dataclasses import asdict, dataclass
from pathlib import Path

from .instruments import INSTRUMENTS, get_instrument

LONG = "long"
SHORT = "short"
SIDE_LABELS = {LONG: "ロング", SHORT: "ショート"}
DATA_VERSION = 1


class PortfolioError(Exception):
    pass


def calc_pnl(side: str, entry_price: float, exit_price: float, qty: int) -> float:
    diff = exit_price - entry_price if side == LONG else entry_price - exit_price
    # 浮動小数点の誤差 (例: 22220.000000000004) を落とすため銭単位で丸める
    return round(diff * qty, 2)


@dataclass
class Position:
    """保有中の建玉。"""

    id: int
    symbol: str
    side: str
    qty: int
    entry_price: float
    opened_at: str  # "YYYY-MM-DD HH:MM" (現在値で約定) または "YYYY-MM-DD" (終値で約定)

    def unrealized_pnl(self, price: float) -> float:
        return calc_pnl(self.side, self.entry_price, price, self.qty)


@dataclass
class Trade:
    """決済済みの取引。一部決済した場合は決済ごとに 1 件。"""

    position_id: int
    symbol: str
    side: str
    qty: int
    entry_price: float
    exit_price: float
    opened_at: str
    closed_at: str

    @property
    def pnl(self) -> float:
        return calc_pnl(self.side, self.entry_price, self.exit_price, self.qty)


@dataclass
class SummaryRow:
    """銘柄ごとの損益集計。"""

    symbol: str
    trades: int  # 決済回数
    wins: int  # 利益が出た決済の回数
    realized: float  # 実現損益
    open_positions: int  # 保有中の建玉数
    unrealized: float | None  # 評価損益 (現在値が取得できなかった場合は None)

    @property
    def win_rate(self) -> float | None:
        return self.wins / self.trades if self.trades else None

    @property
    def total(self) -> float | None:
        return None if self.unrealized is None else round(self.realized + self.unrealized, 2)


def _check_qty(qty: object) -> None:
    if isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0:
        raise PortfolioError(f"数量は 1 以上の整数で指定してください: {qty}")


def _check_price(price: float) -> None:
    if not price > 0:
        raise PortfolioError(f"価格が不正です: {price}")


class Portfolio:
    def __init__(
        self,
        positions: list[Position] | None = None,
        trades: list[Trade] | None = None,
        next_id: int = 1,
    ) -> None:
        self.positions = positions if positions is not None else []
        self.trades = trades if trades is not None else []
        self.next_id = next_id

    # ---- 保存・読み込み ------------------------------------------------

    @classmethod
    def load(cls, path: str | os.PathLike) -> Portfolio:
        path = Path(path)
        if not path.exists():
            return cls()
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            positions = [Position(**p) for p in data.get("positions", [])]
            trades = [Trade(**t) for t in data.get("trades", [])]
            next_id = int(data.get("next_id", 1))
        except (OSError, ValueError, TypeError, AttributeError) as e:
            raise PortfolioError(f"データファイル {path} を読み込めません: {e}") from e
        return cls(positions, trades, next_id)

    def save(self, path: str | os.PathLike) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        data = {
            "version": DATA_VERSION,
            "next_id": self.next_id,
            "positions": [asdict(p) for p in self.positions],
            "trades": [asdict(t) for t in self.trades],
        }
        # 書き込み途中で中断してもデータが壊れないよう、一時ファイルに書いてから置き換える
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        os.replace(tmp, path)

    # ---- 取引 ----------------------------------------------------------

    def open(self, symbol: str, side: str, qty: int, price: float, at: str) -> Position:
        """新規に建玉を作る。"""
        get_instrument(symbol)
        if side not in SIDE_LABELS:
            raise PortfolioError(f"売買区分が不正です: {side}")
        _check_qty(qty)
        _check_price(price)
        position = Position(self.next_id, symbol, side, qty, price, at)
        self.next_id += 1
        self.positions.append(position)
        return position

    def find(self, position_id: int) -> Position:
        for position in self.positions:
            if position.id == position_id:
                return position
        raise PortfolioError(f"建玉 #{position_id} は見つかりません (保有中の建玉は positions で確認できます)。")

    def close(self, position_id: int, price: float, at: str, qty: int | None = None) -> Trade:
        """建玉を決済する。qty を指定すると一部だけ決済する。"""
        position = self.find(position_id)
        qty = position.qty if qty is None else qty
        _check_qty(qty)
        _check_price(price)
        if qty > position.qty:
            raise PortfolioError(f"決済数量 {qty:,} が建玉 #{position_id} の数量 {position.qty:,} を超えています。")
        if at[:10] < position.opened_at[:10]:
            raise PortfolioError(
                f"決済日 {at[:10]} が建玉 #{position_id} の約定日 {position.opened_at[:10]} より前です。"
            )

        trade = Trade(
            position.id,
            position.symbol,
            position.side,
            qty,
            position.entry_price,
            price,
            position.opened_at,
            at,
        )
        self.trades.append(trade)
        position.qty -= qty
        if position.qty == 0:
            self.positions.remove(position)
        return trade

    # ---- 集計 ----------------------------------------------------------

    @property
    def realized_pnl(self) -> float:
        return round(sum(t.pnl for t in self.trades), 2)

    def summary(self, prices: dict[str, float | None]) -> list[SummaryRow]:
        """銘柄ごとの損益。prices は銘柄 ID -> 現在値 (保有中の銘柄の分だけあればよい)。"""
        rows = []
        for inst in INSTRUMENTS:
            trades = [t for t in self.trades if t.symbol == inst.key]
            positions = [p for p in self.positions if p.symbol == inst.key]
            price = prices.get(inst.key)
            if not positions:
                unrealized: float | None = 0.0
            elif price is None:
                unrealized = None
            else:
                unrealized = round(sum(p.unrealized_pnl(price) for p in positions), 2)
            rows.append(
                SummaryRow(
                    symbol=inst.key,
                    trades=len(trades),
                    wins=sum(1 for t in trades if t.pnl > 0),
                    realized=round(sum(t.pnl for t in trades), 2),
                    open_positions=len(positions),
                    unrealized=unrealized,
                )
            )
        return rows
''',
    'trade_sim/prices.py': r'''"""yfinance を使った価格の取得。"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Protocol

from .instruments import Instrument

# 指定日が休場日だったときに、直前の営業日を探してさかのぼる日数
LOOKBACK_DAYS = 14

# 最新の価格を使い回す秒数 (Colab のように開きっぱなしで使う場合も、これより古い価格は取得し直す)
LATEST_CACHE_SECONDS = 60


class PriceError(Exception):
    """価格を取得できなかった。"""


@dataclass(frozen=True)
class Quote:
    price: float
    as_of: date  # 価格の日付 (日足の日付)
    prev_close: float | None = None  # 前営業日の終値


class PriceSource(Protocol):
    def latest(self, inst: Instrument) -> Quote:
        """最新の価格 (取引時間中は現在値、時間外は直近の終値)。"""

    def on_date(self, inst: Instrument, day: date) -> Quote:
        """指定日の終値。休場日なら直前の営業日の終値。"""


class YFinanceSource:
    """Yahoo Finance の日足データを yfinance で取得する。

    日足の最後の行は取引時間中なら当日の途中経過 (現在値) になる。
    東証銘柄は Yahoo Finance 側でおよそ 20 分遅れのデータになる。
    """

    def __init__(self, cache_seconds: float = LATEST_CACHE_SECONDS) -> None:
        self.cache_seconds = cache_seconds
        self._latest_cache: dict[str, tuple[float, Quote]] = {}

    def clear_cache(self) -> None:
        self._latest_cache.clear()

    def latest(self, inst: Instrument) -> Quote:
        cached = self._latest_cache.get(inst.key)
        if cached is not None and time.monotonic() - cached[0] < self.cache_seconds:
            return cached[1]
        closes = self._daily_closes(inst, period="1mo")
        if not closes:
            raise PriceError(f"{inst.name}({inst.ticker}) の価格データを取得できませんでした。")
        day, price = closes[-1]
        prev_close = closes[-2][1] if len(closes) >= 2 else None
        quote = Quote(price, day, prev_close)
        self._latest_cache[inst.key] = (time.monotonic(), quote)
        return quote

    def on_date(self, inst: Instrument, day: date) -> Quote:
        start = day - timedelta(days=LOOKBACK_DAYS)
        end = day + timedelta(days=1)  # yfinance の end はその日を含まない
        closes = [
            (d, p)
            for d, p in self._daily_closes(inst, start=start.isoformat(), end=end.isoformat())
            if d <= day
        ]
        if not closes:
            raise PriceError(
                f"{day} 以前 {LOOKBACK_DAYS} 日間に {inst.name}({inst.ticker}) の価格データがありません。"
            )
        found_day, price = closes[-1]
        return Quote(price, found_day)

    def _daily_closes(self, inst: Instrument, **kwargs) -> list[tuple[date, float]]:
        try:
            import yfinance as yf
        except ImportError as e:
            raise PriceError(
                "yfinance がインストールされていません。`pip install -r requirements.txt` を実行してください。"
            ) from e

        # データが無いときの yfinance 自身のログは、こちらのエラーメッセージで代わりに伝える
        logging.getLogger("yfinance").setLevel(logging.CRITICAL)
        try:
            # auto_adjust=False: 配当で調整されていない、実際の終値を使う
            df = yf.Ticker(inst.ticker).history(interval="1d", auto_adjust=False, actions=False, **kwargs)
        except Exception as e:  # 通信エラーやレート制限など、yfinance は色々な例外を投げる
            raise PriceError(f"{inst.name}({inst.ticker}) の価格取得に失敗しました: {e}") from e

        if df is None or df.empty or "Close" not in df.columns:
            return []
        closes = df["Close"].dropna()
        return [(ts.date(), round(float(value), inst.price_decimals)) for ts, value in closes.items()]
''',
    'trade_sim/simulator.py': r'''"""シミュレーターの操作。コマンドライン (cli.py) と Colab の画面 (notebook.py) の共通部分。

各操作は表示用の Result を返し、表示のしかたは呼び出し側が決める。
"""

from __future__ import annotations

from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from .instruments import INSTRUMENTS, Instrument, UnknownInstrumentError, find_instrument, get_instrument
from .portfolio import SIDE_LABELS, Portfolio, PortfolioError
from .prices import PriceError, PriceSource, YFinanceSource

# 日本に夏時間はないので、固定の +9 時間で日本時間になる
JST = timezone(timedelta(hours=9), "JST")


class InputError(Exception):
    """入力内容の誤り。"""


# 利用者に内容をそのまま伝えればよいエラー
USER_ERRORS = (InputError, PortfolioError, PriceError, UnknownInstrumentError)


def now_jst() -> datetime:
    """日本時間の現在時刻。Colab のように UTC で動く環境でも、取引時刻を日本時間で記録する。"""
    return datetime.now(JST).replace(tzinfo=None)


def parse_date(text: str | None) -> date | None:
    """"YYYY-MM-DD" を日付にする。空欄なら None。"""
    text = (text or "").strip()
    if not text:
        return None
    try:
        return date.fromisoformat(text.replace("/", "-"))
    except ValueError:
        raise InputError(f"日付は YYYY-MM-DD 形式で指定してください: {text}") from None


# ---- 表示用の整形 --------------------------------------------------------


def fmt_yen(value: float | None) -> str:
    if value is None:
        return "-"
    rounded = round(value)
    return "0円" if rounded == 0 else f"{rounded:+,}円"


def fmt_rate(rate: float | None) -> str:
    return "-" if rate is None else f"{rate * 100:.0f}%"


def fmt_change(inst: Instrument, price: float, prev_close: float | None) -> str:
    if not prev_close:
        return "-"
    diff = price - prev_close
    return f"{diff:+,.{inst.price_decimals}f} ({diff / prev_close * 100:+.2f}%)"


@dataclass
class Table:
    headers: list[str]
    rows: list[list[str]]
    aligns: str  # 列ごとの寄せ方 ("<" 左寄せ / ">" 右寄せ)
    footer: list[list[str]] = field(default_factory=list)


@dataclass
class Result:
    table: Table | None = None
    lines: list[str] = field(default_factory=list)  # 約定の結果や合計など
    notes: list[str] = field(default_factory=list)  # 注意書き (価格を取得できなかった等)
    ok: bool = True


# ---- 操作 ----------------------------------------------------------------


class Simulator:
    def __init__(
        self,
        path: str | Path,
        source: PriceSource | None = None,
        clock: Callable[[], datetime] = now_jst,
    ) -> None:
        self.path = Path(path)
        self.source = source if source is not None else YFinanceSource()
        self.clock = clock

    def load(self) -> Portfolio:
        return Portfolio.load(self.path)

    def refresh_prices(self) -> None:
        """取得済みの価格を捨てて、次の操作で取得し直す。"""
        clear = getattr(self.source, "clear_cache", None)
        if clear is not None:
            clear()

    def quote(self, symbols: Iterable[str] = ()) -> Result:
        instruments = [find_instrument(s) for s in symbols] or list(INSTRUMENTS)
        rows, notes = [], []
        for inst in instruments:
            try:
                quote = self.source.latest(inst)
            except PriceError as e:
                rows.append([inst.name, inst.ticker, "-", "-", "-"])
                notes.append(str(e))
                continue
            rows.append(
                [
                    inst.name,
                    inst.ticker,
                    inst.format_price(quote.price),
                    fmt_change(inst, quote.price, quote.prev_close),
                    quote.as_of.isoformat(),
                ]
            )
        table = Table(["銘柄", "ティッカー", "現在値", "前日比", "日付"], rows, "<<>><")
        return Result(table=table, notes=notes, ok=len(notes) < len(instruments))

    def open(
        self, symbol: str, side: str, qty: int, price: float | None = None, day: date | None = None
    ) -> Result:
        """新規にロング (side="long") またはショート (side="short") で建てる。"""
        inst = find_instrument(symbol)
        if side not in SIDE_LABELS:
            raise InputError(f"売買区分が不正です: {side}")
        price, at, note = self._resolve_price(inst, price, day)
        portfolio = self.load()
        position = portfolio.open(inst.key, side, qty, price, at)
        portfolio.save(self.path)
        return Result(
            lines=[
                f"新規{SIDE_LABELS[side]}: #{position.id} {inst.name} {inst.format_qty(position.qty)}"
                f" @ {inst.format_price(price)}円 ({note})"
            ]
        )

    def close(
        self,
        ids: Sequence[int | str],
        qty: int | None = None,
        price: float | None = None,
        day: date | None = None,
    ) -> Result:
        """建玉を決済する。ids に "all" を渡すとすべての建玉。qty を指定すると一部だけ決済する。"""
        portfolio = self.load()
        if [str(i).lower() for i in ids] == ["all"]:
            targets = [p.id for p in portfolio.positions]
            if not targets:
                return Result(lines=["決済する建玉がありません。"])
        else:
            targets = list(dict.fromkeys(_parse_position_id(i) for i in ids))
        if not targets:
            raise InputError("決済する建玉の ID を指定してください。")
        if len(targets) > 1 and qty is not None:
            raise InputError("一部決済 (数量の指定) は、建玉を 1 つだけ指定したときに使えます。")
        if len(targets) > 1 and price is not None:
            raise InputError("価格の指定は、建玉を 1 つだけ指定したときに使えます。")

        # 先にすべての決済価格を決めてから決済する (途中で失敗したら何も保存しない)
        orders = []
        for position_id in targets:
            inst = get_instrument(portfolio.find(position_id).symbol)
            orders.append((position_id, inst, *self._resolve_price(inst, price, day)))

        lines = []
        for position_id, inst, exit_price, at, note in orders:
            trade = portfolio.close(position_id, exit_price, at, qty)
            lines.append(
                f"決済: #{trade.position_id} {inst.name} {SIDE_LABELS[trade.side]} {inst.format_qty(trade.qty)}"
                f"  建値 {inst.format_price(trade.entry_price)} → 決済値 {inst.format_price(trade.exit_price)}"
                f" ({note})  損益 {fmt_yen(trade.pnl)}"
            )
        portfolio.save(self.path)
        lines.append(f"実現損益の通算: {fmt_yen(portfolio.realized_pnl)}")
        return Result(lines=lines)

    def positions(self) -> Result:
        portfolio = self.load()
        if not portfolio.positions:
            return Result(lines=["保有中の建玉はありません。"])
        prices, notes = self._latest_prices({p.symbol for p in portfolio.positions})
        rows = []
        total = 0.0
        for position in portfolio.positions:
            inst = get_instrument(position.symbol)
            price = prices.get(position.symbol)
            if price is None:
                current, pnl_text = "-", "-"
            else:
                pnl = position.unrealized_pnl(price)
                total += pnl
                current, pnl_text = inst.format_price(price), fmt_yen(pnl)
            rows.append(
                [
                    f"#{position.id}",
                    inst.name,
                    SIDE_LABELS[position.side],
                    inst.format_qty(position.qty),
                    inst.format_price(position.entry_price),
                    current,
                    pnl_text,
                    position.opened_at,
                ]
            )
        headers = ["建玉", "銘柄", "売買", "数量", "建値", "現在値", "評価損益", "約定日時"]
        suffix = " (現在値を取得できなかった建玉を除く)" if notes else ""
        return Result(
            table=Table(headers, rows, "<<<>>>><"),
            lines=[f"評価損益の合計: {fmt_yen(total)}{suffix}"],
            notes=notes,
        )

    def history(self, symbol: str | None = None) -> Result:
        trades = self.load().trades
        if symbol:
            key = find_instrument(symbol).key
            trades = [t for t in trades if t.symbol == key]
        if not trades:
            return Result(lines=["決済済みの取引はまだありません。"])
        rows = []
        for trade in trades:
            inst = get_instrument(trade.symbol)
            rows.append(
                [
                    f"#{trade.position_id}",
                    inst.name,
                    SIDE_LABELS[trade.side],
                    inst.format_qty(trade.qty),
                    inst.format_price(trade.entry_price),
                    inst.format_price(trade.exit_price),
                    fmt_yen(trade.pnl),
                    trade.opened_at,
                    trade.closed_at,
                ]
            )
        headers = ["建玉", "銘柄", "売買", "数量", "建値", "決済値", "損益", "約定日時", "決済日時"]
        return Result(
            table=Table(headers, rows, "<<<>>>><<"),
            lines=[f"実現損益の合計: {fmt_yen(sum(t.pnl for t in trades))} ({len(trades)} 件)"],
        )

    def summary(self) -> Result:
        """銘柄別と通算の損益。通算損益 = 実現損益 + 評価損益。"""
        portfolio = self.load()
        prices, errors = self._latest_prices({p.symbol for p in portfolio.positions})
        rows = portfolio.summary(prices)

        table_rows = [
            [
                get_instrument(r.symbol).name,
                str(r.trades),
                fmt_rate(r.win_rate),
                fmt_yen(r.realized),
                str(r.open_positions),
                fmt_yen(r.unrealized),
                fmt_yen(r.total),
            ]
            for r in rows
        ]
        trades = sum(r.trades for r in rows)
        wins = sum(r.wins for r in rows)
        realized = round(sum(r.realized for r in rows), 2)
        unrealized = round(sum(r.unrealized for r in rows if r.unrealized is not None), 2)
        missing = any(r.unrealized is None for r in rows)
        total = round(realized + unrealized, 2)
        footer = [
            [
                "合計",
                str(trades),
                fmt_rate(wins / trades if trades else None),
                fmt_yen(realized),
                str(sum(r.open_positions for r in rows)),
                "-" if missing else fmt_yen(unrealized),
                "-" if missing else fmt_yen(total),
            ]
        ]
        headers = ["銘柄", "決済回数", "勝率", "実現損益", "建玉数", "評価損益", "合計"]
        notes = []
        if missing:
            notes = ["現在値を取得できなかった銘柄の評価損益は、通算損益に含まれていません。", *errors]
        return Result(
            table=Table(headers, table_rows, "<>>>>>>", footer),
            lines=[f"通算損益: {fmt_yen(total)}  (実現損益 {fmt_yen(realized)} + 評価損益 {fmt_yen(unrealized)})"],
            notes=notes,
        )

    def reset(self) -> Result:
        """建玉と取引履歴をすべて削除する。"""
        if not self.path.exists():
            return Result(lines=[f"データファイル {self.path} はありません。"])
        self.path.unlink()
        return Result(lines=["データを削除しました。"])

    # ---- 内部 ------------------------------------------------------------

    def _resolve_price(self, inst: Instrument, price: float | None, day: date | None) -> tuple[float, str, str]:
        """約定価格を決める。戻り値は (価格, 記録する約定日時, 説明)。"""
        now = self.clock()
        if day is not None and day > now.date():
            raise InputError(f"未来の日付 ({day}) は指定できません。")
        if price is not None:
            if not price > 0:
                raise InputError(f"価格は 0 より大きい値を指定してください: {price}")
            at = day.isoformat() if day else now.strftime("%Y-%m-%d %H:%M")
            return price, at, "指定価格"
        if day is not None:
            quote = self.source.on_date(inst, day)
            if quote.as_of == now.date():
                note = f"当日 {quote.as_of} の最新値"  # 取引時間中なら確定した終値ではない
            elif quote.as_of != day:
                note = f"{day} は取引がないため、直前の営業日 {quote.as_of} の終値"
            else:
                note = f"{quote.as_of} の終値"
            return quote.price, quote.as_of.isoformat(), note
        quote = self.source.latest(inst)
        return quote.price, now.strftime("%Y-%m-%d %H:%M"), f"{quote.as_of} 時点の最新値"

    def _latest_prices(self, symbols: set[str]) -> tuple[dict[str, float | None], list[str]]:
        """保有銘柄の現在値をまとめて取得する。取得できなかった銘柄は None。"""
        prices: dict[str, float | None] = {}
        errors = []
        for inst in INSTRUMENTS:
            if inst.key not in symbols:
                continue
            try:
                prices[inst.key] = self.source.latest(inst).price
            except PriceError as e:
                prices[inst.key] = None
                errors.append(str(e))
        return prices, errors


def _parse_position_id(value: int | str) -> int:
    if isinstance(value, int) and not isinstance(value, bool):
        return value
    try:
        return int(str(value).strip().lstrip("#"))
    except ValueError:
        raise InputError(f"建玉 ID は数字で指定してください: {value}") from None
''',
    'trade_sim/table.py': r'''"""全角文字を含む表を桁をそろえて表示する。"""

from __future__ import annotations

import unicodedata
from collections.abc import Sequence


def text_width(text: str) -> int:
    """端末上での表示幅 (全角は 2、半角は 1)。"""
    return sum(2 if unicodedata.east_asian_width(ch) in ("W", "F") else 1 for ch in text)


def _pad(text: str, width: int, align: str) -> str:
    gap = " " * (width - text_width(text))
    return gap + text if align == ">" else text + gap


def render_table(
    headers: Sequence[str],
    rows: Sequence[Sequence[str]],
    aligns: str,
    footer: Sequence[Sequence[str]] = (),
) -> str:
    """aligns は列ごとの寄せ方 ("<" 左寄せ / ">" 右寄せ) を並べた文字列。"""
    all_rows = [headers, *rows, *footer]
    widths = [max(text_width(row[i]) for row in all_rows) for i in range(len(headers))]

    def line(cells: Sequence[str]) -> str:
        return "  ".join(_pad(c, w, a) for c, w, a in zip(cells, widths, aligns)).rstrip()

    separator = "  ".join("-" * w for w in widths)
    lines = [line(headers), separator, *(line(r) for r in rows)]
    if footer:
        lines += [separator, *(line(r) for r in footer)]
    return "\n".join(lines)
''',
}

_package_dir = Path.cwd() / "trade_sim_app"
for _name, _text in _SOURCES.items():
    _path = _package_dir / _name
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_text, encoding="utf-8")
if str(_package_dir) not in sys.path:
    sys.path.insert(0, str(_package_dir))
for _module in [m for m in sys.modules if m == "trade_sim" or m.startswith("trade_sim.")]:
    del sys.modules[_module]  # セルを実行し直したときに、書き出したコードを読み込み直す

# 取引データの保存先
DATA_FILE = str(Path.cwd() / "portfolio.json")
if save_to_drive:
    try:
        from google.colab import drive
    except ImportError:  # Colab 以外 (手元の Jupyter など) ではカレントディレクトリに保存する
        pass
    else:
        drive.mount("/content/drive")
        DATA_FILE = "/content/drive/MyDrive/trade_sim/portfolio.json"
else:
    print("※ 取引データはこのランタイムの中だけに保存されます (ランタイムが終了すると消えます)。")

print("準備ができました。取引データの保存先:", DATA_FILE)

In [ ]:
#@title ② 取引画面を表示 { display-mode: "form" }
#@markdown 下に表示される画面で注文・決済ができます。「価格を更新」を押すと最新の価格を取り直します。
from trade_sim.notebook import show

_app = show(DATA_FILE)

In [ ]:
#@title （必要なときだけ）取引データをすべて削除 { display-mode: "form" }
#@markdown 建玉と取引履歴をすべて消して、最初からやり直します。**delete_all** にチェックを入れてから、このセルを実行してください。
delete_all = False  #@param {type:"boolean"}

from trade_sim.simulator import Simulator

if delete_all:
    print(*Simulator(DATA_FILE).reset().lines)
else:
    print("削除するときは delete_all にチェックを入れてから、このセルを実行してください。")